# P24 — SO Metrics Recompute (uniform 5-seed primary metrics)

**Purpose.** Fill the one gap in the SO evaluation tables: the RED / GS edge policies and the
post-hoc precision variants (fp16 / int8 / pruning) had **accuracy only** and, for post-hoc,
**seed 42 only**. This inference-only pass recomputes accuracy, macro-F1, macro-precision, and
macro-recall for all of them **across the 5 seeds {1,2,3,5,42}**, so every SO technique reports
the same metric set. No training; reuses the existing `full_s*` / `tiny_s*` checkpoints.

**Non-destructive & resumable.** New script `p24/so_metrics_recompute.py`; writes only
`results/p24_so_metrics__s<seed>.json` (per seed) and `results/p24_so_metrics_summary.json`.
One seed per call, skips seeds already done. Estimated ~30–60 min total on GPU.


> **Final_PROJECT note (updated 2026-09-26).** All paths now come from
> `Final_PROJECT/project_paths.py` via the first code cell below — the only thing a new user may
> need to edit is the DAGHAR data path in that file (and `FINAL_PROJECT_ROOT` below, only if the
> folder isn't found automatically). **Saved cell outputs are from the original thesis run**, when
> this code lived at `/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression`; they are kept as the record of that run.
> All training scripts are **resumable**: re-running this notebook in place skips any fold whose
> result files already exist, so it will not overwrite the thesis results.
>
> This notebook's cells were saved **without outputs**, but its result files (`results/p24_so_metrics__s*.json`, `results/p24_so_metrics_summary.json`) are present — running it again only re-prints/re-aggregates them (per-seed passes are skipped when their file exists).

## 1. Mount & paths


In [ ]:
# ── Mount Drive & load Final_PROJECT paths (single source: Final_PROJECT/project_paths.py) ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob

# EDIT ONLY IF you uploaded Final_PROJECT somewhere else on your Drive
# (if this path doesn't exist, the folder is searched for automatically).
FINAL_PROJECT_ROOT = "/content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT"

if not os.path.isfile(os.path.join(FINAL_PROJECT_ROOT, "project_paths.py")):
    hits = []
    for depth in range(0, 5):
        hits = glob.glob("/content/drive/MyDrive/" + "*/" * depth + "Final_PROJECT/project_paths.py")
        if hits: break
    assert hits, "Final_PROJECT not found on Drive — set FINAL_PROJECT_ROOT above"
    FINAL_PROJECT_ROOT = os.path.dirname(hits[0]); print("auto-detected Final_PROJECT at", FINAL_PROJECT_ROOT)
sys.path.insert(0, FINAL_PROJECT_ROOT)
import project_paths as PP            # the DAGHAR data path is set inside project_paths.py

PROJECT_PATH = PP.activate("p24")   # exports DATA_PATH, checks data, cd's into this component
DATA_PATH    = PP.DATA_PATH
P21_SWEEP    = PP.P21_SWEEP        # per-seed frozen ConvNet base_s* live here
os.environ["P21_SWEEP"] = P21_SWEEP  # used as "$P21_SWEEP" in the shell cells below
assert os.path.isdir(P21_SWEEP), "missing P21_SWEEP"

## 2. Setup


In [ ]:
import setup_colab
setup_colab.install_deps(); assert setup_colab.verify(), "fix DATA_PATH"

## 3. Recompute per seed (RED/GS edges + post-hoc precision)
Runs one seed at a time and skips any already finished, so a disconnect only re-runs what's left.


In [ ]:
for s in (1, 2, 3, 5, 42):
    print(f"\n===== seed {s} =====")
    !python -m p24.so_metrics_recompute --data_path "$DATA_PATH" --p21_sweep "$P21_SWEEP" --seed {s}

## 4. Aggregate → 5-seed summary tables
Writes `results/p24_so_metrics_summary.json` with mean±std over the completed seeds and prints
the fused acc / F1 / precision / recall for every edge policy and post-hoc variant.


In [ ]:
!python -m p24.so_metrics_recompute --summary

## 5. Done
Share back `results/p24_so_metrics_summary.json` (and the per-seed files). Combined with the
already-complete NC / AW / capacity metrics, every SO technique will then have a uniform 5-seed
accuracy / macro-F1 / precision / recall set for the thesis tables.
